In [ ]:
!pip install spotipy scikit-learn pandas numpy kagglehub --quiet

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import kagglehub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 15.6 MB/s eta 0:00:00


In [ ]:
client_id = "37e36362ad0f4b009be016da95f96684"
client_secret = "1f61454f1c4140dc8c8e6ab757b308a2"

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=client_id,
    client_secret=client_secret
))

In [ ]:
path = kagglehub.dataset_download("melissamonfared/spotify-tracks-attributes-and-popularity")
print("Dataset downloaded at:", path)

# Load the CSV file
df = pd.read_csv(path + "/dataset.csv")
df.head()

In [ ]:
df = df.dropna()

# Select important features
features = ['danceability', 'energy', 'speechiness', 'acousticness',
            'instrumentalness', 'liveness', 'valence', 'tempo']

# Create target label
df['hit'] = df['popularity'].apply(lambda x: 1 if x > 60 else 0)

X = df[features]
y = df['hit']

# Balance dataset (equal hits and flops)
hit_count = df['hit'].value_counts().min()
df_balanced = pd.concat([
    df[df['hit'] == 1].sample(hit_count, random_state=42),
    df[df['hit'] == 0].sample(hit_count, random_state=42)
])
X = df_balanced[features]
y = df_balanced['hit']

print("Dataset balanced. Shape:", X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = np.round(lr.predict(X_test_scaled))

print("Linear Regression Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
print("Model Comparison:")
print(f"Linear Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"Random Forest Accuracy:     {accuracy_score(y_test, y_pred_rf):.3f}")

In [ ]:
def get_song_features(song_name):
    # Search for up to 5 matching tracks
    results = sp.search(q=song_name, limit=5, type='track')

    tracks = results['tracks']['items']
    if not tracks:
        print("No songs found for:", song_name)
        return None

    # Display all found options
    print(f"\nFound multiple songs for '{song_name}':")
    for i, t in enumerate(tracks):
        print(f"{i+1}. {t['name']} – {t['artists'][0]['name']} (Album: {t['album']['name']})")

    # Ask user to choose
    try:
        choice = int(input("Enter number of the correct song (1–5): ")) - 1
        if choice < 0 or choice >= len(tracks):
            print("Invalid choice. Defaulting to first result.")
            choice = 0
    except:
        print("Invalid input. Defaulting to first result.")
        choice = 0

    # Get chosen song
    track = tracks[choice]
    track_id = track['id']

    # Fetch audio features
    features = sp.audio_features(track_id)[0]
    feature_list = ['danceability', 'energy', 'speechiness', 'acousticness',
                    'instrumentalness', 'liveness', 'valence', 'tempo']

    song_data = [features[f] for f in feature_list]

    print(f"Selected: {track['name']} – {track['artists'][0]['name']}")
    return np.array(song_data).reshape(1, -1)

In [ ]:
def predict_song(song_name):
    sample = get_song_features(song_name)
    if sample is None:
        return

    sample_scaled = scaler.transform(sample)

    lr_pred = "Hit" if lr.predict(sample_scaled)[0] > 0.5 else "Flop"
    rf_pred = "Hit" if rf.predict(sample)[0] == 1 else "Flop"

    print(f"\n🎵 Song: {song_name}")
    print(f"Linear Regression Prediction: {lr_pred}")
    print(f"Random Forest Prediction:     {rf_pred}")

In [ ]:
import pandas as pd

test_songs = pd.DataFrame([
    {   # 1. Likely Hit — upbeat, catchy, energetic
        'danceability': 0.85,
        'energy': 0.9,
        'speechiness': 0.05,
        'acousticness': 0.1,
        'instrumentalness': 0.0,
        'liveness': 0.1,
        'valence': 0.9,
        'tempo': 130
    },
    {   # 2. Likely Flop — slow, low energy, sad tone
        'danceability': 0.35,
        'energy': 0.3,
        'speechiness': 0.05,
        'acousticness': 0.8,
        'instrumentalness': 0.2,
        'liveness': 0.1,
        'valence': 0.2,
        'tempo': 70
    },
    {   # 3. Mid-level — moderate values all around
        'danceability': 0.6,
        'energy': 0.6,
        'speechiness': 0.1,
        'acousticness': 0.3,
        'instrumentalness': 0.05,
        'liveness': 0.2,
        'valence': 0.5,
        'tempo': 200
    }
])[scaler.feature_names_in_]

In [ ]:
# Scale input
test_scaled = scaler.transform(test_songs)

# Linear Regression predictions
lin_preds = ["Hit   " if p > 0.5 else "Flop   " for p in lr.predict(test_scaled)]

# Random Forest predictions
rf_preds = ["Hit   " if p == 1 else "Flop   " for p in rf.predict(test_songs)]

# Combine and display neatly
results = pd.DataFrame({
    "Case": ["Sample 1", "Sample 2", "Sample 3"],
    "   Linear Regression   ": lin_preds,
    "   Random Forest   ": rf_preds
})

print(results)